In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkContext,SparkConf

In [2]:
import os
import shutil

In [3]:
import warnings

In [4]:
warnings.filterwarnings('ignore')

In [5]:
conf=SparkConf().setAppName('RDD_LAB_2').setMaster('local[*]')
sc=SparkContext.getOrCreate(conf=conf)
sc.setLogLevel('Error')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/19 12:02:29 WARN Utils: Your hostname, Samriddhas-Macbook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.1.9.18 instead (on interface en0)
26/05/19 12:02:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 12:02:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/19 12:02:31 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [6]:
print('Spark Version',sc.version)
print('App Name:',sc.appName)

Spark Version 4.1.1
App Name: RDD_LAB_2


In [7]:
numbers=[10,25,4,38,17,52,6,91,3,44]

In [8]:
rdd_nums=sc.parallelize(numbers)

In [9]:
print("Type:",type(rdd_nums))
print("Partition Count:",rdd_nums.getNumPartitions())
print('Count:',rdd_nums.count())
print('First 5 data:',rdd_nums.take(5))

Type: <class 'pyspark.core.rdd.RDD'>
Partition Count: 8


[Stage 0:>                                                          (0 + 8) / 8]

Count: 10
First 5 data: [10, 25, 4, 38, 17]


In [10]:
employees=[
    "Ram,Engineering,72000",
    "Shyam,Doctor,80000",
    "Samriddha,AIEngineer,75000",
    "Krishna,DairyOwner,60000",
    "Gita,HR,49000",
    "Adam,Engineering,95000",
    "Keshab,SystemDesigner,80000",
    "Sandip,AIEngineer,92000",
    "Hiro,Engineering,79000"
]

In [11]:
rdd_emp=sc.parallelize(employees)

In [12]:
print('Employee Count:',rdd_emp.count())
print('First Record:',rdd_emp.first())

Employee Count: 9
First Record: Ram,Engineering,72000


In [13]:
rdd_range=sc.range(1,21)

In [14]:
rdd_range.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

In [15]:
print('Range Sum:',rdd_range.sum())

Range Sum: 210


In [16]:
emp_path='lab_outuput1/employees' #hamle ja kam gariracham, tya bhitra lab_output_1 folder bancha ani bhitra employees folder bhitra bascha
if os.path.exists(emp_path):
    shutil.rmtree(emp_path)

In [17]:
rdd_emp.saveAsTextFile(emp_path)
print("Employees data saved to:",emp_path)

Employees data saved to: lab_outuput1/employees


In [18]:
files=os.listdir(emp_path)
part_files=[f for f in files if f.startswith('part')]
print('files written:',files)
print('number of part files:',len(part_files))

files written: ['.part-00006.crc', '._SUCCESS.crc', '.part-00007.crc', '.part-00005.crc', '.part-00004.crc', '.part-00000.crc', '.part-00001.crc', 'part-00005', 'part-00002', '.part-00003.crc', 'part-00003', '.part-00002.crc', 'part-00004', '_SUCCESS', 'part-00001', 'part-00006', 'part-00007', 'part-00000']
number of part files: 8


In [19]:
#now to load all data in bulk

In [21]:
rdd_emp_loaded=sc.textFile('lab_outuput1/employees')
rdd_emp_loaded.collect()

['Adam,Engineering,95000',
 'Samriddha,AIEngineer,75000',
 'Krishna,DairyOwner,60000',
 'Gita,HR,49000',
 'Shyam,Doctor,80000',
 'Keshab,SystemDesigner,80000',
 'Sandip,AIEngineer,92000',
 'Hiro,Engineering,79000',
 'Ram,Engineering,72000']

In [22]:
def parse_row(line):
    parts=line.split(',')
    name=parts[0]
    dept=parts[1]
    salary=int(parts[2])
    return (name,dept,salary)

In [23]:
rdd_parsed=rdd_emp_loaded.map(parse_row)
rdd_parsed.collect()

[('Adam', 'Engineering', 95000),
 ('Samriddha', 'AIEngineer', 75000),
 ('Krishna', 'DairyOwner', 60000),
 ('Gita', 'HR', 49000),
 ('Shyam', 'Doctor', 80000),
 ('Keshab', 'SystemDesigner', 80000),
 ('Sandip', 'AIEngineer', 92000),
 ('Hiro', 'Engineering', 79000),
 ('Ram', 'Engineering', 72000)]

In [24]:
rdd_eng=rdd_parsed.filter(lambda r:r[1]=="Engineering")
print("Engineering Count:",rdd_eng.count())

Engineering Count: 3


In [27]:
rdd_eng=rdd_parsed.filter(lambda r:r[1]=="Engineering")
print("Engineering Students' Names:")
for i in rdd_eng.collect():
    print(i[0])

Engineering Students' Names:
Adam
Hiro
Ram


In [31]:
filtered=rdd_parsed.filter(lambda x:x[2]>70000)
print(filtered.collect())

[('Adam', 'Engineering', 95000), ('Samriddha', 'AIEngineer', 75000), ('Shyam', 'Doctor', 80000), ('Keshab', 'SystemDesigner', 80000), ('Sandip', 'AIEngineer', 92000), ('Hiro', 'Engineering', 79000), ('Ram', 'Engineering', 72000)]
